# ClaimGuard AI — Reviewed Colab Prototype

Run from top to bottom in **Google Colab**. Uses synthetic data only. The 4/12/24-hour SLA targets are illustrative, and rule-based recommendations are not trained AI. The final dashboard is an educational demo, not a production claims system.

**Note:** Running the initialization cells rebuilds `claimguard.db` from synthetic data; download a backup before rerunning if you have made manual changes.

In [1]:

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
random.seed(42)
np.random.seed(42)

print("ClaimGuard AI environment initialized successfully!")


ClaimGuard AI environment initialized successfully!


In [2]:

NUM_CLAIMS = 1000

providers = [
    "Dallas General Hospital",
    "North Texas Medical Center",
    "Lone Star Healthcare",
    "Metro Family Clinic",
    "Richardson Health Center"
]

insurance_companies = [
    "Aetna",
    "Cigna",
    "UnitedHealthcare",
    "Blue Cross Blue Shield",
    "Humana"
]

procedure_codes = [
    "99213", "99214", "99203",
    "80053", "85025", "93000"
]

diagnosis_codes = [
    "I10", "E11.9", "J06.9",
    "M54.50", "R51.9"
]

claims = []

for i in range(NUM_CLAIMS):

    submitted_at = datetime(2026, 10, 1) + timedelta(
        minutes=random.randint(0, 20000)
    )

    claim = {
        "claim_id": f"CLM{100001+i}",
        "patient_id": f"PAT{random.randint(1000,9999)}",
        "provider": random.choice(providers),
        "insurance_company": random.choice(insurance_companies),
        "procedure_code": random.choice(procedure_codes),
        "diagnosis_code": random.choice(diagnosis_codes),
        "claim_amount": round(random.uniform(50, 5000), 2),
        "submission_date": submitted_at,
        "status": "RECEIVED"
    }

    claims.append(claim)

df = pd.DataFrame(claims)

print("Total claims generated:", len(df))

display(df.head(10))


Total claims generated: 1000


,claim_id,patient_id,provider,insurance_company,procedure_code,diagnosis_code,claim_amount,submission_date,status
0,CLM100001,PAT1409,Lone Star Healthcare,Cigna,99214,E11.9,3695.53,2026-10-03 12:48:00,RECEIVED
1,CLM100002,PAT2424,Richardson Health Center,Blue Cross Blue Shield,99213,I10,513.79,2026-10-13 09:50:00,RECEIVED
2,CLM100003,PAT9279,Richardson Health Center,Aetna,85025,E11.9,3594.30,2026-10-06 07:03:00,RECEIVED
3,CLM100004,PAT7873,North Texas Medical Center,Blue Cross Blue Shield,85025,J06.9,4056.68,2026-10-13 09:36:00,RECEIVED
4,CLM100005,PAT3615,Metro Family Clinic,UnitedHealthcare,99203,E11.9,1115.80,2026-10-01 03:32:00,RECEIVED
5,CLM100006,PAT2674,Dallas General Hospital,Blue Cross Blue Shield,99213,J06.9,4245.10,2026-10-08 15:49:00,RECEIVED
6,CLM100007,PAT5333,Dallas General Hospital,Blue Cross Blue Shield,85025,I10,4866.92,2026-10-14 17:42:00,RECEIVED
7,CLM100008,PAT2291,Richardson Health Center,UnitedHealthcare,93000,R51.9,4432.99,2026-10-09 14:43:00,RECEIVED
8,CLM100009,PAT4150,Dallas General Hospital,Aetna,93000,E11.9,3876.69,2026-10-09 05:30:00,RECEIVED
9,CLM100010,PAT4814,Dallas General Hospital,Blue Cross Blue Shield,99203,M54.50,3196.64,2026-10-02 19:34:00,RECEIVED


In [3]:

# Work on a separate copy
df_faulty = df.copy()

# Inject distinct types of problems into different rows
missing_patient_rows = df_faulty.index[0:25]
invalid_amount_rows = df_faulty.index[25:50]
invalid_procedure_rows = df_faulty.index[50:75]
duplicate_claim_rows = df_faulty.index[75:100]

# 1. Missing patient identifiers
df_faulty.loc[
    missing_patient_rows, "patient_id"
] = None

# 2. Negative claim amounts
df_faulty.loc[
    invalid_amount_rows, "claim_amount"
] = -100.00

# 3. Invalid procedure codes
df_faulty.loc[
    invalid_procedure_rows, "procedure_code"
] = "INVALID"

# 4. Duplicate existing claim IDs
df_faulty.loc[
    duplicate_claim_rows, "claim_id"
] = df_faulty.loc[100:124, "claim_id"].values

print("Total input records:", len(df_faulty))
print("Synthetic errors injected:", 100)

display(df_faulty.head(10))


Total input records: 1000
Synthetic errors injected: 100


,claim_id,patient_id,provider,insurance_company,procedure_code,diagnosis_code,claim_amount,submission_date,status
0,CLM100001,None,Lone Star Healthcare,Cigna,99214,E11.9,3695.53,2026-10-03 12:48:00,RECEIVED
1,CLM100002,None,Richardson Health Center,Blue Cross Blue Shield,99213,I10,513.79,2026-10-13 09:50:00,RECEIVED
2,CLM100003,None,Richardson Health Center,Aetna,85025,E11.9,3594.30,2026-10-06 07:03:00,RECEIVED
3,CLM100004,None,North Texas Medical Center,Blue Cross Blue Shield,85025,J06.9,4056.68,2026-10-13 09:36:00,RECEIVED
4,CLM100005,None,Metro Family Clinic,UnitedHealthcare,99203,E11.9,1115.80,2026-10-01 03:32:00,RECEIVED
5,CLM100006,None,Dallas General Hospital,Blue Cross Blue Shield,99213,J06.9,4245.10,2026-10-08 15:49:00,RECEIVED
6,CLM100007,None,Dallas General Hospital,Blue Cross Blue Shield,85025,I10,4866.92,2026-10-14 17:42:00,RECEIVED
7,CLM100008,None,Richardson Health Center,UnitedHealthcare,93000,R51.9,4432.99,2026-10-09 14:43:00,RECEIVED
8,CLM100009,None,Dallas General Hospital,Aetna,93000,E11.9,3876.69,2026-10-09 05:30:00,RECEIVED
9,CLM100010,None,Dallas General Hospital,Blue Cross Blue Shield,99203,M54.50,3196.64,2026-10-02 19:34:00,RECEIVED


In [4]:

# Export dataset
file_name = "claimguard_synthetic_claims.csv"

df_faulty.to_csv(file_name, index=False)

print("Dataset saved successfully:", file_name)
print("Total rows:", len(df_faulty))
print("Total columns:", len(df_faulty.columns))

# Download to your computer

print("CSV created:", file_name)


Dataset saved successfully: claimguard_synthetic_claims.csv
Total rows: 1000
Total columns: 9
CSV created: claimguard_synthetic_claims.csv


In [5]:

# Build simple validation rules
df_faulty["error_reason"] = ""

valid_procedure_codes = set(procedure_codes)

# Mark the earlier occurrence of deliberately duplicated IDs
duplicate_mask = df_faulty["claim_id"].duplicated(keep="last")

for index, row in df_faulty.iterrows():

    errors = []

    if pd.isna(row["patient_id"]):
        errors.append("Missing Patient ID")

    if row["claim_amount"] <= 0:
        errors.append("Invalid Claim Amount")

    if row["procedure_code"] not in valid_procedure_codes:
        errors.append("Invalid Procedure Code")

    if duplicate_mask.loc[index]:
        errors.append("Duplicate Claim ID")

    df_faulty.at[index, "error_reason"] = "; ".join(errors)

    df_faulty.at[index, "status"] = (
        "FAILED" if errors else "VALIDATED"
    )

# Summarize validation results
print("CLAIMGUARD AI - VALIDATION REPORT")
print("--------------------------------")

print("Total Claims:", len(df_faulty))

print(
    "Validated Claims:",
    (df_faulty["status"] == "VALIDATED").sum()
)

print(
    "Failed Claims:",
    (df_faulty["status"] == "FAILED").sum()
)

display(
    df_faulty[df_faulty["status"] == "FAILED"].head(10)
)

# Automated quality checks
assert len(df_faulty) == 1000
assert (df_faulty["status"] == "FAILED").sum() == 100
assert (df_faulty["status"] == "VALIDATED").sum() == 900

print("All initial validation tests passed!")


CLAIMGUARD AI - VALIDATION REPORT
--------------------------------
Total Claims: 1000
Validated Claims: 900
Failed Claims: 100


,claim_id,patient_id,provider,insurance_company,procedure_code,diagnosis_code,claim_amount,submission_date,status,error_reason
0,CLM100001,None,Lone Star Healthcare,Cigna,99214,E11.9,3695.53,2026-10-03 12:48:00,FAILED,Missing Patient ID
1,CLM100002,None,Richardson Health Center,Blue Cross Blue Shield,99213,I10,513.79,2026-10-13 09:50:00,FAILED,Missing Patient ID
2,CLM100003,None,Richardson Health Center,Aetna,85025,E11.9,3594.30,2026-10-06 07:03:00,FAILED,Missing Patient ID
3,CLM100004,None,North Texas Medical Center,Blue Cross Blue Shield,85025,J06.9,4056.68,2026-10-13 09:36:00,FAILED,Missing Patient ID
4,CLM100005,None,Metro Family Clinic,UnitedHealthcare,99203,E11.9,1115.80,2026-10-01 03:32:00,FAILED,Missing Patient ID
5,CLM100006,None,Dallas General Hospital,Blue Cross Blue Shield,99213,J06.9,4245.10,2026-10-08 15:49:00,FAILED,Missing Patient ID
6,CLM100007,None,Dallas General Hospital,Blue Cross Blue Shield,85025,I10,4866.92,2026-10-14 17:42:00,FAILED,Missing Patient ID
7,CLM100008,None,Richardson Health Center,UnitedHealthcare,93000,R51.9,4432.99,2026-10-09 14:43:00,FAILED,Missing Patient ID
8,CLM100009,None,Dallas General Hospital,Aetna,93000,E11.9,3876.69,2026-10-09 05:30:00,FAILED,Missing Patient ID
9,CLM100010,None,Dallas General Hospital,Blue Cross Blue Shield,99203,M54.50,3196.64,2026-10-02 19:34:00,FAILED,Missing Patient ID


All initial validation tests passed!


In [6]:

df_faulty.to_csv(
    "claimguard_validation_results.csv",
    index=False
)

print("Validation CSV saved.")

print("Validation results exported successfully!")


Validation CSV saved.
Validation results exported successfully!


In [7]:

import sqlite3

# Connect to our project database
conn = sqlite3.connect("claimguard.db")
conn.execute("PRAGMA foreign_keys = ON")

# Save validation results into SQL
df_faulty.to_sql(
    "claims",
    conn,
    if_exists="replace",
    index=False
)

print("Claims successfully stored in SQL database!")

# Verify the data
result = pd.read_sql_query(
    "SELECT status, COUNT(*) AS total FROM claims GROUP BY status",
    conn
)

display(result)


Claims successfully stored in SQL database!


,status,total
0,FAILED,100
1,VALIDATED,900


In [8]:

# STEP 8: Automated Incident Ticket Generation

from datetime import datetime
import pandas as pd

# Retrieve failed claim records from SQL
failed_claims = pd.read_sql_query("""
    SELECT *
    FROM claims
    WHERE status = 'FAILED'
""", conn)

tickets = []

for i, row in failed_claims.iterrows():

    error = row["error_reason"]

    # Assign priority based on error type
    if "Duplicate Claim ID" in error:
        priority = "HIGH"
    elif "Invalid Claim Amount" in error:
        priority = "HIGH"
    elif "Invalid Procedure Code" in error:
        priority = "MEDIUM"
    else:
        priority = "LOW"

    tickets.append({
        "ticket_id": f"INC{100001 + len(tickets)}",
        "claim_id": row["claim_id"],
        "error_reason": error,
        "priority": priority,
        "ticket_status": "OPEN",
        "assigned_team": "Support & Monitoring",
        "created_at": datetime.now().isoformat()
    })

# Convert tickets to DataFrame
tickets_df = pd.DataFrame(tickets)

# Save tickets to SQL database
tickets_df.to_sql(
    "incidents",
    conn,
    if_exists="replace",
    index=False
)

print("CLAIMGUARD AI - INCIDENT SYSTEM")
print("--------------------------------")
print("Total support tickets created:", len(tickets_df))

display(tickets_df.head(10))


CLAIMGUARD AI - INCIDENT SYSTEM
--------------------------------
Total support tickets created: 100


,ticket_id,claim_id,error_reason,priority,ticket_status,assigned_team,created_at
0,INC100001,CLM100001,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.912836
1,INC100002,CLM100002,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913030
2,INC100003,CLM100003,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913148
3,INC100004,CLM100004,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913246
4,INC100005,CLM100005,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913334
5,INC100006,CLM100006,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913423
6,INC100007,CLM100007,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913520
7,INC100008,CLM100008,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913604
8,INC100009,CLM100009,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913687
9,INC100010,CLM100010,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:44.913742


In [9]:

# STEP 9: SQL Incident Monitoring Report

report = pd.read_sql_query("""
    SELECT
        priority,
        ticket_status,
        COUNT(*) AS total_incidents
    FROM incidents
    GROUP BY priority, ticket_status
    ORDER BY
        CASE priority
            WHEN 'HIGH' THEN 1
            WHEN 'MEDIUM' THEN 2
            WHEN 'LOW' THEN 3
        END
""", conn)

print("CLAIMGUARD AI - INCIDENT MONITORING REPORT")
display(report)

# Verify all failed claims generated tickets
assert len(tickets_df) == 100
assert tickets_df["ticket_id"].is_unique

print("Incident generation tests passed!")


CLAIMGUARD AI - INCIDENT MONITORING REPORT


,priority,ticket_status,total_incidents
0,HIGH,OPEN,50
1,MEDIUM,OPEN,25
2,LOW,OPEN,25


Incident generation tests passed!


In [10]:
import plotly.express as px
import plotly.graph_objects as go
print("Plotly ready")

Plotly ready


In [11]:

# Load real data from our SQL database
claim_summary = pd.read_sql_query("""
    SELECT status, COUNT(*) AS total
    FROM claims
    GROUP BY status
""", conn)

fig = px.pie(
    claim_summary,
    values="total",
    names="status",
    hole=0.6,
    title="Healthcare Claims Validation Overview",
    color="status",
    color_discrete_map={
        "VALIDATED": "#16A34A",
        "FAILED": "#DC2626"
    }
)

fig.update_layout(
    template="plotly_dark",
    height=450,
    title_x=0.5
)

fig.show()


In [12]:

incident_summary = pd.read_sql_query("""
    SELECT priority, COUNT(*) AS total
    FROM incidents
    GROUP BY priority
""", conn)

fig = px.bar(
    incident_summary,
    x="priority",
    y="total",
    color="priority",
    title="Support Incident Priority Distribution",
    category_orders={
        "priority": ["HIGH", "MEDIUM", "LOW"]
    },
    color_discrete_map={
        "HIGH": "#DC2626",
        "MEDIUM": "#F59E0B",
        "LOW": "#3B82F6"
    },
    text="total"
)

fig.update_layout(
    template="plotly_dark",
    height=450,
    title_x=0.5,
    xaxis_title="Incident Priority",
    yaxis_title="Number of Incidents",
    showlegend=False
)

fig.show()


In [13]:

# STEP 14: Add unique internal claim record identifiers

claims_upgrade = pd.read_sql_query(
    "SELECT * FROM claims ORDER BY rowid",
    conn
)

# Each transaction receives its own internal ID
claims_upgrade.insert(
    0,
    "record_id",
    range(1, len(claims_upgrade) + 1)
)

# Replace claims table with an explicit primary key
conn.execute("DROP TABLE IF EXISTS claims_new")

conn.execute("""
CREATE TABLE claims_new (
    record_id INTEGER PRIMARY KEY,
    claim_id TEXT,
    patient_id TEXT,
    provider TEXT,
    insurance_company TEXT,
    procedure_code TEXT,
    diagnosis_code TEXT,
    claim_amount REAL,
    submission_date TEXT,
    status TEXT,
    error_reason TEXT
)
""")

claims_upgrade.to_sql(
    "claims_new",
    conn,
    if_exists="append",
    index=False
)

conn.execute("DROP TABLE claims")
conn.execute("ALTER TABLE claims_new RENAME TO claims")
conn.commit()

print("Claims database upgraded successfully!")
print("Unique records:", len(claims_upgrade))
print("Unique internal IDs:", claims_upgrade["record_id"].nunique())


Claims database upgraded successfully!
Unique records: 1000
Unique internal IDs: 1000


In [14]:

# STEP 15: Link support tickets using record_id

failed_records = pd.read_sql_query("""
    SELECT record_id, claim_id, error_reason
    FROM claims
    WHERE status = 'FAILED'
    ORDER BY record_id
""", conn)

incidents_upgrade = []

for idx, row in failed_records.iterrows():

    reason = row["error_reason"]

    if "Duplicate Claim ID" in reason:
        priority = "HIGH"
    elif "Invalid Claim Amount" in reason:
        priority = "HIGH"
    elif "Invalid Procedure Code" in reason:
        priority = "MEDIUM"
    else:
        priority = "LOW"

    incidents_upgrade.append({
        "ticket_id": f"INC{100001 + len(incidents_upgrade)}",
        "record_id": int(row["record_id"]),
        "claim_id": row["claim_id"],
        "error_reason": reason,
        "priority": priority,
        "ticket_status": "OPEN",
        "assigned_team": "Support & Monitoring",
        "created_at": datetime.now().isoformat(),
        "resolved_at": None
    })

incidents_df = pd.DataFrame(incidents_upgrade)

incidents_df.to_sql(
    "incidents",
    conn,
    if_exists="replace",
    index=False
)

print("Incidents upgraded:", len(incidents_df))
print("Each incident now links to a unique claim record.")

assert incidents_df["record_id"].is_unique
assert len(incidents_df) == 100

display(incidents_df.head())


Incidents upgraded: 100
Each incident now links to a unique claim record.


,ticket_id,record_id,claim_id,error_reason,priority,ticket_status,assigned_team,created_at,resolved_at
0,INC100001,1,CLM100001,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:57.614283,None
1,INC100002,2,CLM100002,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:57.614444,None
2,INC100003,3,CLM100003,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:57.614573,None
3,INC100004,4,CLM100004,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:57.614714,None
4,INC100005,5,CLM100005,Missing Patient ID,LOW,OPEN,Support & Monitoring,2026-10-19T17:30:57.614825,None


In [15]:

# STEP 16: Investigate a support incident

def investigate_incident(ticket_id):

    query = """
        SELECT
            i.ticket_id,
            i.record_id,
            i.priority,
            i.ticket_status,
            i.error_reason,
            c.claim_id,
            c.patient_id,
            c.provider,
            c.insurance_company,
            c.procedure_code,
            c.claim_amount,
            c.status AS claim_status
        FROM incidents i
        JOIN claims c
            ON i.record_id = c.record_id
        WHERE i.ticket_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(ticket_id,)
    )

    if result.empty:
        print("Ticket not found.")
        return

    print("CLAIMGUARD AI — INCIDENT INVESTIGATION")
    display(result.T)


# Test with a real incident
investigate_incident("INC100001")


CLAIMGUARD AI — INCIDENT INVESTIGATION


,0
ticket_id,INC100001
record_id,1
priority,LOW
ticket_status,OPEN
error_reason,Missing Patient ID
claim_id,CLM100001
patient_id,None
provider,Lone Star Healthcare
insurance_company,Cigna
procedure_code,99214


In [16]:

# STEP 17: Create incident audit log table

conn.execute("""
CREATE TABLE IF NOT EXISTS incident_audit_log (
    audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticket_id TEXT NOT NULL,
    action TEXT NOT NULL,
    performed_by TEXT NOT NULL,
    action_time TEXT NOT NULL,
    notes TEXT
)
""")

conn.commit()

print("Incident audit log created successfully!")


Incident audit log created successfully!


In [17]:

# STEP 18: Resolve incident with audit tracking

def resolve_incident(ticket_id, engineer, notes):

    # Check the ticket exists and is still open
    ticket = conn.execute("""
        SELECT ticket_status
        FROM incidents
        WHERE ticket_id = ?
    """, (ticket_id,)).fetchone()

    if ticket is None:
        print("Ticket not found.")
        return

    if ticket[0] != "OPEN":
        print("Ticket is not open.")
        return

    resolution_time = datetime.now().isoformat()

    with conn:
        conn.execute("""
            UPDATE incidents
            SET ticket_status = 'RESOLVED',
                resolved_at = ?
            WHERE ticket_id = ?
        """, (resolution_time, ticket_id))

        conn.execute("""
            INSERT INTO incident_audit_log
            (ticket_id, action, performed_by,
             action_time, notes)
            VALUES (?, ?, ?, ?, ?)
        """, (
            ticket_id,
            "INCIDENT_RESOLVED",
            engineer,
            resolution_time,
            notes
        ))

    print(f"Incident {ticket_id} resolved successfully!")
    print("Audit record created.")


# Resolve one sample incident
resolve_incident(
    "INC100001",
    "Support Engineer",
    "Missing patient identifier investigated; ticket closed for demonstration."
)


Incident INC100001 resolved successfully!
Audit record created.


In [18]:

# STEP 19: Verify incident status and audit history

print("INCIDENT STATUS")

display(pd.read_sql_query("""
    SELECT ticket_id, record_id,
           ticket_status, resolved_at
    FROM incidents
    WHERE ticket_id = 'INC100001'
""", conn))

print("AUDIT HISTORY")

display(pd.read_sql_query("""
    SELECT *
    FROM incident_audit_log
    WHERE ticket_id = 'INC100001'
""", conn))


INCIDENT STATUS


,ticket_id,record_id,ticket_status,resolved_at
0,INC100001,1,RESOLVED,2026-10-19T17:30:57.846173


AUDIT HISTORY


,audit_id,ticket_id,action,performed_by,action_time,notes
0,1,INC100001,INCIDENT_RESOLVED,Support Engineer,2026-10-19T17:30:57.846173,Missing patient identifier investigated; ticke...


In [19]:

# STEP 20: Smart Recovery Policy Engine

def recommend_recovery(error_reason):

    error = str(error_reason).lower()

    if "duplicate claim id" in error:
        return {
            "category": "DUPLICATE_TRANSACTION",
            "strategy": "MANUAL_REVIEW",
            "recommendation": (
                "Investigate the original claim and "
                "check whether this is an accidental "
                "duplicate or a legitimate resubmission."
            )
        }

    elif "invalid claim amount" in error:
        return {
            "category": "INVALID_AMOUNT",
            "strategy": "MANUAL_CORRECTION",
            "recommendation": (
                "Verify the billed amount against the "
                "source record before correcting it."
            )
        }

    elif "invalid procedure code" in error:
        return {
            "category": "INVALID_PROCEDURE",
            "strategy": "MANUAL_CORRECTION",
            "recommendation": (
                "Confirm the correct procedure code "
                "from the authorized source."
            )
        }

    elif "missing patient id" in error:
        return {
            "category": "MISSING_INFORMATION",
            "strategy": "MANUAL_CORRECTION",
            "recommendation": (
                "Request the missing patient identifier "
                "from the authorized source."
            )
        }

    elif "timeout" in error:
        return {
            "category": "TEMPORARY_FAILURE",
            "strategy": "AUTO_RETRY",
            "recommendation": (
                "Retry with bounded attempts and "
                "idempotency protection."
            )
        }

    else:
        return {
            "category": "UNKNOWN",
            "strategy": "ESCALATE",
            "recommendation": (
                "Escalate for further investigation."
            )
        }


print("Recovery policy engine initialized!")


Recovery policy engine initialized!


In [20]:
# STEP 21: Generate recovery recommendations for support tickets
incidents_data = pd.read_sql_query("SELECT * FROM incidents", conn)
recovery_results = []
for _, incident in incidents_data.iterrows():
    recommendation = recommend_recovery(incident["error_reason"])
    recovery_results.append({
        "ticket_id": incident["ticket_id"],
        "record_id": int(incident["record_id"]),
        "category": recommendation["category"],
        "strategy": recommendation["strategy"],
        "recommendation": recommendation["recommendation"]
    })
recovery_df = pd.DataFrame(recovery_results)
recovery_df.to_sql("recovery_recommendations", conn, if_exists="replace", index=False)
print("Recovery recommendations:",len(recovery_df))
display(recovery_df.head())

Recovery recommendations: 100


,ticket_id,record_id,category,strategy,recommendation
0,INC100001,1,MISSING_INFORMATION,MANUAL_CORRECTION,Request the missing patient identifier from th...
1,INC100002,2,MISSING_INFORMATION,MANUAL_CORRECTION,Request the missing patient identifier from th...
2,INC100003,3,MISSING_INFORMATION,MANUAL_CORRECTION,Request the missing patient identifier from th...
3,INC100004,4,MISSING_INFORMATION,MANUAL_CORRECTION,Request the missing patient identifier from th...
4,INC100005,5,MISSING_INFORMATION,MANUAL_CORRECTION,Request the missing patient identifier from th...


In [21]:
# STEP 22: Create replay history BEFORE running replay operations
conn.execute("""
CREATE TABLE IF NOT EXISTS replay_history (
    replay_id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticket_id TEXT NOT NULL,
    record_id INTEGER NOT NULL,
    replay_status TEXT NOT NULL,
    attempted_at TEXT NOT NULL,
    attempted_by TEXT NOT NULL,
    notes TEXT
)
""")
conn.commit()
print("Replay history ready")

Replay history ready


In [22]:

# STEP 23: Safe Manual Correction and Transaction Replay

def correct_and_replay(ticket_id, field, new_value,
                       engineer="Support Engineer"):

    allowed_fields = {
        "patient_id",
        "claim_amount",
        "procedure_code"
    }

    if field not in allowed_fields:
        print("Correction denied: field is not permitted.")
        return

    ticket = conn.execute("""
        SELECT record_id, ticket_status
        FROM incidents
        WHERE ticket_id = ?
    """, (ticket_id,)).fetchone()

    if ticket is None:
        print("Ticket not found.")
        return

    record_id, ticket_status = ticket

    if ticket_status != "OPEN":
        print("Only OPEN tickets can be replayed.")
        return

    claim = conn.execute("""
        SELECT claim_id, patient_id, claim_amount,
               procedure_code
        FROM claims
        WHERE record_id = ?
    """, (record_id,)).fetchone()

    if claim is None:
        print("Claim record not found.")
        return

    claim_id, patient_id, amount, procedure = claim

    # Proposed correction (not yet committed)
    proposed = {
        "patient_id": patient_id,
        "claim_amount": amount,
        "procedure_code": procedure
    }

    proposed[field] = new_value

    errors = []

    if pd.isna(proposed["patient_id"]) or not str(
        proposed["patient_id"]
    ).strip():
        errors.append("Missing Patient ID")

    try:
        corrected_amount = float(
            proposed["claim_amount"]
        )
        if not np.isfinite(corrected_amount) or corrected_amount <= 0:
            errors.append("Invalid Claim Amount")
    except (ValueError, TypeError):
        errors.append("Invalid Claim Amount")

    if proposed["procedure_code"] not in procedure_codes:
        errors.append("Invalid Procedure Code")

    # Duplicate claim IDs require manual investigation.
    # Do not replay a claim with a non-unique claim ID.
    duplicate_count = conn.execute("""
        SELECT COUNT(*)
        FROM claims
        WHERE claim_id = ?
    """, (claim_id,)).fetchone()[0]

    if duplicate_count > 1:
        errors.append("Duplicate Claim ID")

    replay_time = datetime.now().isoformat()

    if errors:
        with conn:
            conn.execute("""
                INSERT INTO replay_history
                (ticket_id, record_id, replay_status,
                 attempted_at, attempted_by, notes)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (
                ticket_id, record_id, "REJECTED",
                replay_time, engineer,
                "; ".join(errors)
            ))

        print("Replay rejected.")
        print("Remaining validation errors:", errors)
        return

    # Only commit after the entire proposed record
    # passes validation.
    with conn:
        conn.execute(
            f"UPDATE claims SET {field} = ? WHERE record_id = ?",
            (new_value, record_id)
        )

        conn.execute("""
            UPDATE claims
            SET status = 'VALIDATED',
                error_reason = ''
            WHERE record_id = ?
        """, (record_id,))

        conn.execute("""
            UPDATE incidents
            SET ticket_status = 'RESOLVED',
                resolved_at = ?
            WHERE ticket_id = ?
        """, (replay_time, ticket_id))

        conn.execute("""
            INSERT INTO replay_history
            (ticket_id, record_id, replay_status,
             attempted_at, attempted_by, notes)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            ticket_id, record_id, "SUCCESS",
            replay_time, engineer,
            f"Validated correction to {field}"
        ))

        conn.execute("""
            INSERT INTO incident_audit_log
            (ticket_id, action, performed_by,
             action_time, notes)
            VALUES (?, ?, ?, ?, ?)
        """, (
            ticket_id,
            "CLAIM_CORRECTED_AND_VALIDATED",
            engineer,
            replay_time,
            f"Corrected {field}; validation passed."
        ))

    print("Claim successfully corrected and revalidated!")
    print("Incident resolved and audit record created.")


In [23]:

# STEP 24: Inspect before correction

investigate_incident("INC100026")

print("Recovery recommendation:")
print(
    recommend_recovery("Invalid Claim Amount")
)


CLAIMGUARD AI — INCIDENT INVESTIGATION


,0
ticket_id,INC100026
record_id,26
priority,HIGH
ticket_status,OPEN
error_reason,Invalid Claim Amount
claim_id,CLM100026
patient_id,PAT3705
provider,Lone Star Healthcare
insurance_company,Humana
procedure_code,85025


Recovery recommendation:
{'category': 'INVALID_AMOUNT', 'strategy': 'MANUAL_CORRECTION', 'recommendation': 'Verify the billed amount against the source record before correcting it.'}


In [24]:

correct_and_replay(
    ticket_id="INC100026",
    field="claim_amount",
    new_value=250.00,
    engineer="Demo Support Engineer"
)


Claim successfully corrected and revalidated!
Incident resolved and audit record created.


In [25]:
# STEP 25: Verify corrected claim, incident, replay, and audit
for name, query in [
    ("Claim", "SELECT record_id, claim_id, claim_amount, status, error_reason FROM claims WHERE record_id=26"),
    ("Incident", "SELECT ticket_id, ticket_status, resolved_at FROM incidents WHERE ticket_id='INC100026'"),
    ("Replay", "SELECT * FROM replay_history WHERE ticket_id='INC100026'"),
    ("Audit", "SELECT * FROM incident_audit_log WHERE ticket_id='INC100026'")
]:
    print(name)
    display(pd.read_sql_query(query, conn))
assert conn.execute("SELECT status FROM claims WHERE record_id=26").fetchone()[0] == 'VALIDATED'
assert conn.execute("SELECT ticket_status FROM incidents WHERE ticket_id='INC100026'").fetchone()[0] == 'RESOLVED'
assert conn.execute("SELECT COUNT(*) FROM replay_history WHERE ticket_id='INC100026' AND replay_status='SUCCESS'").fetchone()[0] == 1
print('Recovery checks passed')

Claim


,record_id,claim_id,claim_amount,status,error_reason
0,26,CLM100026,250.0,VALIDATED,


Incident


,ticket_id,ticket_status,resolved_at
0,INC100026,RESOLVED,2026-10-19T17:30:58.327508


Replay


,replay_id,ticket_id,record_id,replay_status,attempted_at,attempted_by,notes
0,1,INC100026,26,SUCCESS,2026-10-19T17:30:58.327508,Demo Support Engineer,Validated correction to claim_amount


Audit


,audit_id,ticket_id,action,performed_by,action_time,notes
0,2,INC100026,CLAIM_CORRECTED_AND_VALIDATED,Demo Support Engineer,2026-10-19T17:30:58.327508,Corrected claim_amount; validation passed.


Recovery checks passed


In [26]:
# STEP 26: Updated status report
claims_status = pd.read_sql_query("SELECT status, COUNT(*) AS total FROM claims GROUP BY status", conn)
tickets_status = pd.read_sql_query("SELECT ticket_status, COUNT(*) AS total FROM incidents GROUP BY ticket_status", conn)
print("Claims:")
display(claims_status)
print("Tickets:")
display(tickets_status)
# Two tickets are closed in this demonstration, but only one claim has been corrected.
assert dict(zip(claims_status.status,claims_status.total)) == {'FAILED': 99, 'VALIDATED': 901}
assert dict(zip(tickets_status.ticket_status,tickets_status.total)) == {'OPEN': 98, 'RESOLVED': 2}
print('All end-to-end checks passed')

Claims:


,status,total
0,FAILED,99
1,VALIDATED,901


Tickets:


,ticket_status,total
0,OPEN,98
1,RESOLVED,2


All end-to-end checks passed


In [27]:

# STEP 27: Initialize Incident Intelligence

import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px

from datetime import datetime, timedelta, timezone

# Load claims and incidents from our SQL database
claims_data = pd.read_sql_query("""
    SELECT * FROM claims
""", conn)

incidents_data = pd.read_sql_query("""
    SELECT * FROM incidents
""", conn)

print("CLAIMGUARD AI - INCIDENT INTELLIGENCE")
print("-----------------------------------")
print("Claims loaded:", len(claims_data))
print("Incidents loaded:", len(incidents_data))

# Ensure critical tables and columns exist
assert "record_id" in claims_data.columns
assert "record_id" in incidents_data.columns
assert "created_at" in incidents_data.columns
assert "resolved_at" in incidents_data.columns

print("Intelligence system initialized successfully!")


CLAIMGUARD AI - INCIDENT INTELLIGENCE
-----------------------------------
Claims loaded: 1000
Incidents loaded: 100
Intelligence system initialized successfully!


In [28]:

# STEP 28: Explainable Root Cause Analysis

def analyze_root_cause(ticket_id):

    query = """
        SELECT
            i.ticket_id,
            i.record_id,
            i.error_reason,
            i.priority,
            i.ticket_status,
            c.claim_id,
            c.provider,
            c.insurance_company,
            c.claim_amount,
            c.procedure_code,
            c.patient_id
        FROM incidents i
        JOIN claims c
            ON i.record_id = c.record_id
        WHERE i.ticket_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(ticket_id,)
    )

    if result.empty:
        print("Incident not found.")
        return None

    incident = result.iloc[0]

    # Use the original incident error,
    # which remains available after a correction.
    analysis = recommend_recovery(
        incident["error_reason"]
    )

    report = {
        "ticket_id": ticket_id,
        "claim_id": incident["claim_id"],
        "provider": incident["provider"],
        "insurance": incident["insurance_company"],
        "priority": incident["priority"],
        "status": incident["ticket_status"],
        "detected_issue": incident["error_reason"],
        "root_cause_category": analysis["category"],
        "recovery_strategy": analysis["strategy"],
        "recommended_action": analysis["recommendation"]
    }

    print("CLAIMGUARD AI - ROOT CAUSE REPORT")
    print("---------------------------------")

    for key, value in report.items():
        print(f"{key}: {value}")

    return report


# Test using an existing incident
root_cause_report = analyze_root_cause("INC100027")


CLAIMGUARD AI - ROOT CAUSE REPORT
---------------------------------
ticket_id: INC100027
claim_id: CLM100027
provider: Lone Star Healthcare
insurance: Blue Cross Blue Shield
priority: HIGH
status: OPEN
detected_issue: Invalid Claim Amount
root_cause_category: INVALID_AMOUNT
recovery_strategy: MANUAL_CORRECTION
recommended_action: Verify the billed amount against the source record before correcting it.


In [29]:

# STEP 29: SLA Monitoring Engine

SLA_HOURS = {
    "HIGH": 4,
    "MEDIUM": 12,
    "LOW": 24
}

sla_data = pd.read_sql_query("""
    SELECT
        ticket_id,
        record_id,
        priority,
        ticket_status,
        created_at,
        resolved_at
    FROM incidents
""", conn)

# Parse timestamps consistently in UTC
sla_data["created_at"] = pd.to_datetime(
    sla_data["created_at"],
    utc=True
)

sla_data["resolved_at"] = pd.to_datetime(
    sla_data["resolved_at"],
    utc=True
)

now = pd.Timestamp.now(tz="UTC")

sla_data["sla_hours"] = (
    sla_data["priority"].map(SLA_HOURS)
)

sla_data["deadline"] = (
    sla_data["created_at"]
    + pd.to_timedelta(sla_data["sla_hours"], unit="h")
)

def calculate_sla_status(row):

    if row["ticket_status"] == "RESOLVED":

        if pd.isna(row["resolved_at"]):
            return "MISSING_RESOLUTION_TIME"

        if row["resolved_at"] <= row["deadline"]:
            return "RESOLVED_ON_TIME"

        return "RESOLVED_LATE"

    if row["ticket_status"] != "OPEN":
        return "NOT_APPLICABLE"

    hours_remaining = (
        row["deadline"] - now
    ).total_seconds() / 3600

    if hours_remaining < 0:
        return "BREACHED"

    elif hours_remaining <= 1:
        return "AT_RISK"

    else:
        return "WITHIN_SLA"


sla_data["sla_status"] = sla_data.apply(
    calculate_sla_status,
    axis=1
)

print("SLA MONITORING RESULTS")

display(
    sla_data[
        ["ticket_id", "priority", "deadline",
         "ticket_status", "sla_status"]
    ].head(15)
)

print("\nSLA SUMMARY")
display(
    sla_data["sla_status"]
    .value_counts()
    .rename_axis("sla_status")
    .reset_index(name="count")
)


SLA MONITORING RESULTS


,ticket_id,priority,deadline,ticket_status,sla_status
0,INC100001,LOW,2026-10-20 17:30:57.614283+00:00,RESOLVED,RESOLVED_ON_TIME
1,INC100002,LOW,2026-10-20 17:30:57.614444+00:00,OPEN,WITHIN_SLA
2,INC100003,LOW,2026-10-20 17:30:57.614573+00:00,OPEN,WITHIN_SLA
3,INC100004,LOW,2026-10-20 17:30:57.614714+00:00,OPEN,WITHIN_SLA
4,INC100005,LOW,2026-10-20 17:30:57.614825+00:00,OPEN,WITHIN_SLA
5,INC100006,LOW,2026-10-20 17:30:57.614933+00:00,OPEN,WITHIN_SLA
6,INC100007,LOW,2026-10-20 17:30:57.617108+00:00,OPEN,WITHIN_SLA
7,INC100008,LOW,2026-10-20 17:30:57.617293+00:00,OPEN,WITHIN_SLA
8,INC100009,LOW,2026-10-20 17:30:57.617410+00:00,OPEN,WITHIN_SLA
9,INC100010,LOW,2026-10-20 17:30:57.617520+00:00,OPEN,WITHIN_SLA



SLA SUMMARY


,sla_status,count
0,WITHIN_SLA,98
1,RESOLVED_ON_TIME,2


In [30]:

# STEP 30: SLA Monitoring Dashboard

sla_summary = (
    sla_data["sla_status"]
    .value_counts()
    .reset_index()
)

sla_summary.columns = [
    "sla_status", "incident_count"
]

fig = px.bar(
    sla_summary,
    x="sla_status",
    y="incident_count",
    color="sla_status",
    text="incident_count",
    title="ClaimGuard AI - Incident SLA Monitoring",
    color_discrete_map={
        "BREACHED": "#DC2626",
        "AT_RISK": "#F59E0B",
        "WITHIN_SLA": "#16A34A",
        "RESOLVED_ON_TIME": "#2563EB",
        "RESOLVED_LATE": "#9333EA"
    }
)

fig.update_layout(
    template="plotly_dark",
    height=450,
    xaxis_title="SLA Status",
    yaxis_title="Number of Incidents",
    showlegend=False
)

fig.show()


In [31]:

# STEP 31: Provider and Insurance Failure Analysis

provider_analysis = pd.read_sql_query("""
    SELECT
        provider,
        COUNT(*) AS total_claims,
        SUM(CASE
            WHEN status = 'FAILED' THEN 1
            ELSE 0
        END) AS failed_claims,
        ROUND(
            100.0 * SUM(CASE
                WHEN status = 'FAILED' THEN 1
                ELSE 0
            END) / COUNT(*),
            2
        ) AS failure_rate
    FROM claims
    GROUP BY provider
    ORDER BY failure_rate DESC
""", conn)

insurance_analysis = pd.read_sql_query("""
    SELECT
        insurance_company,
        COUNT(*) AS total_claims,
        SUM(CASE
            WHEN status = 'FAILED' THEN 1
            ELSE 0
        END) AS failed_claims,
        ROUND(
            100.0 * SUM(CASE
                WHEN status = 'FAILED' THEN 1
                ELSE 0
            END) / COUNT(*),
            2
        ) AS failure_rate
    FROM claims
    GROUP BY insurance_company
    ORDER BY failure_rate DESC
""", conn)

print("PROVIDER FAILURE ANALYSIS")
display(provider_analysis)

print("INSURANCE FAILURE ANALYSIS")
display(insurance_analysis)

fig = px.bar(
    provider_analysis,
    x="provider",
    y="failure_rate",
    color="failed_claims",
    title="Claim Failure Rate by Healthcare Provider",
    text="failure_rate"
)

fig.update_layout(
    template="plotly_dark",
    height=500,
    xaxis_title="Healthcare Provider",
    yaxis_title="Failure Rate (%)"
)

fig.show()



PROVIDER FAILURE ANALYSIS


,provider,total_claims,failed_claims,failure_rate
0,Lone Star Healthcare,200,22,11.00
1,Dallas General Hospital,195,21,10.77
2,North Texas Medical Center,199,19,9.55
3,Richardson Health Center,210,20,9.52
4,Metro Family Clinic,196,17,8.67


INSURANCE FAILURE ANALYSIS


,insurance_company,total_claims,failed_claims,failure_rate
0,Cigna,207,29,14.01
1,Humana,205,23,11.22
2,Blue Cross Blue Shield,206,22,10.68
3,Aetna,196,14,7.14
4,UnitedHealthcare,186,11,5.91


In [32]:

# STEP 32: Operational Intelligence Report

total_claims = conn.execute(
    "SELECT COUNT(*) FROM claims"
).fetchone()[0]

failed_claims_count = conn.execute("""
    SELECT COUNT(*) FROM claims
    WHERE status = 'FAILED'
""").fetchone()[0]

open_incidents = conn.execute("""
    SELECT COUNT(*) FROM incidents
    WHERE ticket_status = 'OPEN'
""").fetchone()[0]

resolved_incidents = conn.execute("""
    SELECT COUNT(*) FROM incidents
    WHERE ticket_status = 'RESOLVED'
""").fetchone()[0]

# Mean Time to Resolution (resolved incidents only)
resolved_data = sla_data[
    (sla_data["ticket_status"] == "RESOLVED")
    & sla_data["resolved_at"].notna()
].copy()

if len(resolved_data) > 0:
    resolved_data["resolution_hours"] = (
        resolved_data["resolved_at"]
        - resolved_data["created_at"]
    ).dt.total_seconds() / 3600

    mttr = resolved_data["resolution_hours"].mean()
else:
    mttr = None

failure_rate = (
    failed_claims_count / total_claims * 100
    if total_claims else 0
)

breached_count = (
    (sla_data["sla_status"] == "BREACHED").sum()
)

print("=" * 50)
print("CLAIMGUARD AI - OPERATIONAL INTELLIGENCE")
print("=" * 50)

print(f"Total Claims: {total_claims}")
print(f"Failed Claims: {failed_claims_count}")
print(f"Failure Rate: {failure_rate:.2f}%")
print(f"Open Incidents: {open_incidents}")
print(f"Resolved Incidents: {resolved_incidents}")
print(f"Open SLA Breaches: {breached_count}")

if mttr is not None:
    print(f"Mean Time to Resolution: {mttr:.2f} hours")
else:
    print("Mean Time to Resolution: N/A")

print("=" * 50)


CLAIMGUARD AI - OPERATIONAL INTELLIGENCE
Total Claims: 1000
Failed Claims: 99
Failure Rate: 9.90%
Open Incidents: 98
Resolved Incidents: 2
Open SLA Breaches: 0
Mean Time to Resolution: 0.00 hours


In [33]:

# STEP 33: Export Project Reports

provider_analysis.to_csv(
    "provider_failure_analysis.csv",
    index=False
)

insurance_analysis.to_csv(
    "insurance_failure_analysis.csv",
    index=False
)

sla_data.to_csv(
    "incident_sla_report.csv",
    index=False
)

print("ClaimGuard AI reports saved successfully!")

# List generated files
import os

for filename in [
    "provider_failure_analysis.csv",
    "insurance_failure_analysis.csv",
    "incident_sla_report.csv"
]:
    print(filename, "-", os.path.getsize(filename), "bytes")


ClaimGuard AI reports saved successfully!
provider_failure_analysis.csv - 227 bytes
insurance_failure_analysis.csv - 180 bytes
incident_sla_report.csv - 10547 bytes


In [34]:

# STEP 34: Create automated processing tables

import sqlite3
import pandas as pd
import time
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

conn.execute("""
CREATE TABLE IF NOT EXISTS processing_jobs (
    record_id INTEGER PRIMARY KEY,
    scenario TEXT NOT NULL,
    job_status TEXT NOT NULL DEFAULT 'PENDING',
    attempts INTEGER NOT NULL DEFAULT 0,
    last_error TEXT,
    updated_at TEXT NOT NULL,
    FOREIGN KEY (record_id) REFERENCES claims(record_id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS processing_receipts (
    record_id INTEGER PRIMARY KEY,
    processed_at TEXT NOT NULL,
    FOREIGN KEY (record_id) REFERENCES claims(record_id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS processing_attempt_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    record_id INTEGER NOT NULL,
    attempt_number INTEGER NOT NULL,
    outcome TEXT NOT NULL,
    details TEXT,
    attempted_at TEXT NOT NULL
)
""")

conn.commit()
print("Processing and recovery tables created!")


Processing and recovery tables created!


In [35]:

# STEP 35: Prepare simulated processing jobs

validated_records = pd.read_sql_query("""
    SELECT record_id, claim_id
    FROM claims
    WHERE status = 'VALIDATED'
    ORDER BY record_id
    LIMIT 12
""", conn)

assert len(validated_records) == 12

scenarios = [
    "TIMEOUT_THEN_SUCCESS",
    "HTTP_503_THEN_SUCCESS",
    "HTTP_400_PERMANENT",
    "TIMEOUT_AFTER_COMMIT"
] * 3

with conn:
    for (_, claim), scenario in zip(
        validated_records.iterrows(), scenarios
    ):
        conn.execute("""
            INSERT OR IGNORE INTO processing_jobs
            (record_id, scenario, job_status,
             attempts, updated_at)
            VALUES (?, ?, 'PENDING', 0, ?)
        """, (
            int(claim["record_id"]),
            scenario,
            utc_now()
        ))

print("Simulated processing jobs initialized!")

display(pd.read_sql_query("""
    SELECT
        j.record_id,
        c.claim_id,
        j.scenario,
        j.job_status
    FROM processing_jobs j
    JOIN claims c
        ON j.record_id = c.record_id
    ORDER BY j.record_id
""", conn))


Simulated processing jobs initialized!


,record_id,claim_id,scenario,job_status
0,26,CLM100026,TIMEOUT_THEN_SUCCESS,PENDING
1,101,CLM100101,HTTP_503_THEN_SUCCESS,PENDING
2,102,CLM100102,HTTP_400_PERMANENT,PENDING
3,103,CLM100103,TIMEOUT_AFTER_COMMIT,PENDING
4,104,CLM100104,TIMEOUT_THEN_SUCCESS,PENDING
5,105,CLM100105,HTTP_503_THEN_SUCCESS,PENDING
6,106,CLM100106,HTTP_400_PERMANENT,PENDING
7,107,CLM100107,TIMEOUT_AFTER_COMMIT,PENDING
8,108,CLM100108,TIMEOUT_THEN_SUCCESS,PENDING
9,109,CLM100109,HTTP_503_THEN_SUCCESS,PENDING


In [36]:

# STEP 36: Simulated Healthcare Processing API

def simulate_processing_api(record_id, scenario, attempt):

    # Idempotency check:
    # Do not process a transaction that already succeeded.
    existing = conn.execute("""
        SELECT record_id
        FROM processing_receipts
        WHERE record_id = ?
    """, (record_id,)).fetchone()

    if existing:
        return "ALREADY_PROCESSED"

    if scenario == "TIMEOUT_THEN_SUCCESS" and attempt == 1:
        return "TIMEOUT"

    if scenario == "HTTP_503_THEN_SUCCESS" and attempt <= 2:
        return "HTTP_503"

    if scenario == "HTTP_400_PERMANENT":
        return "HTTP_400"

    if scenario == "TIMEOUT_AFTER_COMMIT":
        # Simulate a server processing the transaction,
        # while the response fails to reach the client.
        with conn:
            conn.execute("""
                INSERT OR IGNORE INTO processing_receipts
                (record_id, processed_at)
                VALUES (?, ?)
            """, (record_id, utc_now()))

        return "TIMEOUT_AFTER_COMMIT"

    # Normal successful processing
    with conn:
        conn.execute("""
            INSERT OR IGNORE INTO processing_receipts
            (record_id, processed_at)
            VALUES (?, ?)
        """, (record_id, utc_now()))

    return "SUCCESS"


print("Simulated healthcare processing API ready!")


Simulated healthcare processing API ready!


In [37]:

# STEP 37: Smart Retry Engine

MAX_ATTEMPTS = 3

RETRYABLE_ERRORS = {
    "TIMEOUT",
    "HTTP_503",
    "TIMEOUT_AFTER_COMMIT"
}

def process_with_recovery(record_id):

    job = conn.execute("""
        SELECT scenario, job_status, attempts
        FROM processing_jobs
        WHERE record_id = ?
    """, (record_id,)).fetchone()

    if job is None:
        print("Processing job not found.")
        return

    scenario, status, previous_attempts = job

    if status in ("RECOVERED", "COMPLETED", "ESCALATED"):
        print(f"Claim {record_id}: already finalized ({status})")
        return

    print(f"\nProcessing claim record {record_id}")

    for attempt in range(previous_attempts + 1, MAX_ATTEMPTS + 1):

        # Check receipt before every attempt.
        receipt = conn.execute("""
            SELECT record_id FROM processing_receipts
            WHERE record_id = ?
        """, (record_id,)).fetchone()

        if receipt:
            with conn:
                conn.execute("""
                    UPDATE processing_jobs
                    SET job_status = 'RECOVERED',
                        updated_at = ?
                    WHERE record_id = ?
                """, (utc_now(), record_id))

            print("Existing receipt found; duplicate avoided.")
            return

        outcome = simulate_processing_api(
            record_id, scenario, attempt
        )

        print(f"Attempt {attempt}: {outcome}")

        with conn:
            conn.execute("""
                INSERT INTO processing_attempt_log
                (record_id, attempt_number, outcome,
                 details, attempted_at)
                VALUES (?, ?, ?, ?, ?)
            """, (
                record_id, attempt, outcome,
                f"Scenario: {scenario}",
                utc_now()
            ))

            conn.execute("""
                UPDATE processing_jobs
                SET attempts = ?,
                    last_error = ?,
                    updated_at = ?
                WHERE record_id = ?
            """, (
                attempt,
                None if outcome == "SUCCESS" else outcome,
                utc_now(),
                record_id
            ))

        if outcome in ("SUCCESS", "ALREADY_PROCESSED"):
            with conn:
                conn.execute("""
                    UPDATE processing_jobs
                    SET job_status = 'COMPLETED',
                        updated_at = ?
                    WHERE record_id = ?
                """, (utc_now(), record_id))

            print("Processing completed successfully.")
            return

        # For ambiguous timeouts, check whether
        # processing actually completed.
        if outcome == "TIMEOUT_AFTER_COMMIT":
            receipt = conn.execute("""
                SELECT record_id
                FROM processing_receipts
                WHERE record_id = ?
            """, (record_id,)).fetchone()

            if receipt:
                with conn:
                    conn.execute("""
                        UPDATE processing_jobs
                        SET job_status = 'RECOVERED',
                            updated_at = ?
                        WHERE record_id = ?
                    """, (utc_now(), record_id))

                print("Confirmed existing receipt. No duplicate!")
                return

        if outcome not in RETRYABLE_ERRORS:
            final_status = "ESCALATED"
            break

        if attempt == MAX_ATTEMPTS:
            final_status = "ESCALATED"
            break

        delay_seconds = 2 ** (attempt - 1)
        print(f"Backoff policy: wait {delay_seconds} second(s)")

        # Accelerated waiting for the Colab demonstration
        time.sleep(0.05)

    with conn:
        conn.execute("""
            UPDATE processing_jobs
            SET job_status = ?,
                updated_at = ?
            WHERE record_id = ?
        """, (final_status, utc_now(), record_id))

    print(f"Final status: {final_status}")


print("Smart recovery engine initialized!")


Smart recovery engine initialized!


In [38]:

# STEP 38: Execute the Recovery Engine

jobs_to_process = pd.read_sql_query("""
    SELECT record_id
    FROM processing_jobs
    WHERE job_status = 'PENDING'
    ORDER BY record_id
""", conn)

for record_id in jobs_to_process["record_id"]:
    process_with_recovery(int(record_id))

print("\nCLAIMGUARD AI - RECOVERY EXECUTION FINISHED")



Processing claim record 26
Attempt 1: TIMEOUT
Backoff policy: wait 1 second(s)
Attempt 2: SUCCESS
Processing completed successfully.

Processing claim record 101
Attempt 1: HTTP_503
Backoff policy: wait 1 second(s)
Attempt 2: HTTP_503
Backoff policy: wait 2 second(s)
Attempt 3: SUCCESS
Processing completed successfully.

Processing claim record 102
Attempt 1: HTTP_400
Final status: ESCALATED

Processing claim record 103
Attempt 1: TIMEOUT_AFTER_COMMIT
Confirmed existing receipt. No duplicate!

Processing claim record 104
Attempt 1: TIMEOUT
Backoff policy: wait 1 second(s)
Attempt 2: SUCCESS
Processing completed successfully.

Processing claim record 105
Attempt 1: HTTP_503
Backoff policy: wait 1 second(s)
Attempt 2: HTTP_503
Backoff policy: wait 2 second(s)
Attempt 3: SUCCESS
Processing completed successfully.

Processing claim record 106
Attempt 1: HTTP_400
Final status: ESCALATED

Processing claim record 107
Attempt 1: TIMEOUT_AFTER_COMMIT
Confirmed existing receipt. No duplicate!



In [39]:

# STEP 39: Automated Recovery Monitoring

recovery_summary = pd.read_sql_query("""
    SELECT job_status, COUNT(*) AS total
    FROM processing_jobs
    GROUP BY job_status
""", conn)

print("AUTOMATED PROCESSING RESULTS")
display(recovery_summary)

fig = px.bar(
    recovery_summary,
    x="job_status",
    y="total",
    color="job_status",
    text="total",
    title="ClaimGuard AI - Automated Recovery Results",
    color_discrete_map={
        "COMPLETED": "#16A34A",
        "RECOVERED": "#2563EB",
        "ESCALATED": "#DC2626",
        "PENDING": "#F59E0B"
    }
)

fig.update_layout(
    template="plotly_dark",
    height=450,
    xaxis_title="Processing Outcome",
    yaxis_title="Number of Claims",
    showlegend=False
)

fig.show()


AUTOMATED PROCESSING RESULTS


,job_status,total
0,COMPLETED,6
1,ESCALATED,3
2,RECOVERED,3


In [40]:

# STEP 40: Verify Recovery Safety

print("PROCESSING ATTEMPT HISTORY")

display(pd.read_sql_query("""
    SELECT
        record_id,
        attempt_number,
        outcome,
        attempted_at
    FROM processing_attempt_log
    ORDER BY record_id, attempt_number
""", conn))

print("\nVERIFY IDEMPOTENCY")

duplicate_receipts = pd.read_sql_query("""
    SELECT record_id, COUNT(*) AS total
    FROM processing_receipts
    GROUP BY record_id
    HAVING COUNT(*) > 1
""", conn)

assert duplicate_receipts.empty
print("PASS: No duplicate processing receipts!")

# Validate outcomes
job_summary = pd.read_sql_query("""
    SELECT scenario, job_status, COUNT(*) AS total
    FROM processing_jobs
    GROUP BY scenario, job_status
""", conn)

display(job_summary)

assert conn.execute("""
    SELECT COUNT(*) FROM processing_receipts
""").fetchone()[0] == 9

assert conn.execute("""
    SELECT COUNT(*) FROM processing_jobs
    WHERE job_status = 'ESCALATED'
""").fetchone()[0] == 3

print("All automated recovery tests passed!")


PROCESSING ATTEMPT HISTORY


,record_id,attempt_number,outcome,attempted_at
0,26,1,TIMEOUT,2026-10-19T17:30:59.391603+00:00
1,26,2,SUCCESS,2026-10-19T17:30:59.461271+00:00
2,101,1,HTTP_503,2026-10-19T17:30:59.481346+00:00
3,101,2,HTTP_503,2026-10-19T17:30:59.545081+00:00
4,101,3,SUCCESS,2026-10-19T17:30:59.638211+00:00
5,102,1,HTTP_400,2026-10-19T17:30:59.666202+00:00
6,103,1,TIMEOUT_AFTER_COMMIT,2026-10-19T17:30:59.706281+00:00
7,104,1,TIMEOUT,2026-10-19T17:30:59.729278+00:00
8,104,2,SUCCESS,2026-10-19T17:30:59.806298+00:00
9,105,1,HTTP_503,2026-10-19T17:30:59.826225+00:00



VERIFY IDEMPOTENCY
PASS: No duplicate processing receipts!


,scenario,job_status,total
0,HTTP_400_PERMANENT,ESCALATED,3
1,HTTP_503_THEN_SUCCESS,COMPLETED,3
2,TIMEOUT_AFTER_COMMIT,RECOVERED,3
3,TIMEOUT_THEN_SUCCESS,COMPLETED,3


All automated recovery tests passed!


In [41]:

# STEP 41: Export Recovery Reports

pd.read_sql_query("""
    SELECT * FROM processing_jobs
""", conn).to_csv(
    "claimguard_processing_jobs.csv",
    index=False
)

pd.read_sql_query("""
    SELECT * FROM processing_attempt_log
""", conn).to_csv(
    "claimguard_recovery_history.csv",
    index=False
)

pd.read_sql_query("""
    SELECT * FROM processing_receipts
""", conn).to_csv(
    "claimguard_processing_receipts.csv",
    index=False
)

print("Recovery reports exported successfully!")


Recovery reports exported successfully!


## Unified Support Operations Center

The previous notebook contained multiple experimental Gradio dashboards and debug cells. Those have been consolidated into **one** interface below. Functions open a new SQLite connection per request to avoid callback thread errors.

In [42]:
!pip -q install gradio
import gradio as gr
import plotly.express as px
from pathlib import Path
import sqlite3
DATABASE_PATH = str(Path("claimguard.db").resolve())
print("Gradio:", gr.__version__)

Gradio: 6.30.0


In [43]:

# STEP 58A: Professional Dashboard Analytics

import sqlite3
import pandas as pd
import plotly.express as px
import gradio as gr
from pathlib import Path

DATABASE_PATH = str(Path("claimguard.db").resolve())

def get_professional_dashboard():

    # Open a fresh database connection for Gradio
    with sqlite3.connect(DATABASE_PATH) as db:

        claims = pd.read_sql_query("""
            SELECT record_id, claim_id, provider,
                   insurance_company, claim_amount, status
            FROM claims
            ORDER BY record_id DESC
            LIMIT 100
        """, db)

        incidents = pd.read_sql_query("""
            SELECT ticket_id, claim_id, priority,
                   ticket_status, error_reason
            FROM incidents
            ORDER BY ticket_id DESC
            LIMIT 100
        """, db)

        claim_summary = pd.read_sql_query("""
            SELECT status, COUNT(*) AS total
            FROM claims
            GROUP BY status
        """, db)

        priority_summary = pd.read_sql_query("""
            SELECT priority, COUNT(*) AS total
            FROM incidents
            GROUP BY priority
        """, db)

        provider_summary = pd.read_sql_query("""
            SELECT provider,
                   COUNT(*) AS total_claims,
                   SUM(CASE WHEN status = 'FAILED'
                       THEN 1 ELSE 0 END) AS failed_claims
            FROM claims
            GROUP BY provider
        """, db)

        recovery_summary = pd.read_sql_query("""
            SELECT job_status, COUNT(*) AS total
            FROM processing_jobs
            GROUP BY job_status
        """, db)

        total = db.execute(
            "SELECT COUNT(*) FROM claims"
        ).fetchone()[0]

        validated = db.execute("""
            SELECT COUNT(*) FROM claims
            WHERE status = 'VALIDATED'
        """).fetchone()[0]

        failed = db.execute("""
            SELECT COUNT(*) FROM claims
            WHERE status = 'FAILED'
        """).fetchone()[0]

        open_tickets = db.execute("""
            SELECT COUNT(*) FROM incidents
            WHERE ticket_status = 'OPEN'
        """).fetchone()[0]

    # Build interactive Plotly charts
    claim_chart = px.pie(
        claim_summary,
        names="status",
        values="total",
        hole=0.65,
        title="Healthcare Claims Validation",
        color="status",
        color_discrete_map={
            "VALIDATED": "#10B981",
            "FAILED": "#EF4444"
        }
    )

    priority_chart = px.bar(
        priority_summary,
        x="priority",
        y="total",
        color="priority",
        text="total",
        title="Incident Priority Distribution",
        category_orders={
            "priority": ["HIGH", "MEDIUM", "LOW"]
        },
        color_discrete_map={
            "HIGH": "#EF4444",
            "MEDIUM": "#F59E0B",
            "LOW": "#3B82F6"
        }
    )

    # Actual failure rate, rather than raw failures
    provider_summary["failure_rate"] = (
        100 * provider_summary["failed_claims"]
        / provider_summary["total_claims"]
    ).round(2)

    provider_chart = px.bar(
        provider_summary,
        x="provider",
        y="failure_rate",
        title="Provider Claim Failure Rate",
        text="failure_rate"
    )

    recovery_chart = px.bar(
        recovery_summary,
        x="job_status",
        y="total",
        color="job_status",
        title="Automated Processing Recovery",
        text="total"
    )

    for chart in [
        claim_chart,
        priority_chart,
        provider_chart,
        recovery_chart
    ]:
        chart.update_layout(
            template="plotly_white",
            height=360,
            margin=dict(l=30, r=30, t=55, b=60),
            font=dict(family="Arial")
        )

    return (
        str(total),
        str(validated),
        str(failed),
        str(open_tickets),
        claim_chart,
        priority_chart,
        provider_chart,
        recovery_chart,
        claims,
        incidents
    )


print("Professional dashboard analytics initialized!")

# Test before connecting to Gradio
test_result = get_professional_dashboard()

assert len(test_result) == 10
print("Dashboard test passed!")
print("Current metrics:", test_result[:4])


Professional dashboard analytics initialized!
Dashboard test passed!
Current metrics: ('1000', '901', '99', '98')


In [44]:

# STEP 59A: Claim Search and Investigation Engine

import sqlite3
import pandas as pd
from pathlib import Path

DATABASE_PATH = str(Path("claimguard.db").resolve())

def search_claim(claim_id):

    claim_id = str(claim_id).strip().upper()

    if not claim_id:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            "Please enter a Claim ID."
        )

    with sqlite3.connect(DATABASE_PATH) as db:

        # Search all matching claim records
        claims = pd.read_sql_query("""
            SELECT
                record_id,
                claim_id,
                patient_id,
                provider,
                insurance_company,
                procedure_code,
                diagnosis_code,
                claim_amount,
                status,
                error_reason
            FROM claims
            WHERE UPPER(claim_id) = ?
            ORDER BY record_id
        """, db, params=(claim_id,))

        if claims.empty:
            return (
                claims,
                pd.DataFrame(),
                f"No claim found for {claim_id}."
            )

        # Retrieve tickets using unique internal record IDs
        record_ids = claims["record_id"].tolist()

        placeholders = ",".join(
            ["?"] * len(record_ids)
        )

        tickets = pd.read_sql_query(f"""
            SELECT
                ticket_id,
                record_id,
                priority,
                ticket_status,
                error_reason,
                created_at,
                resolved_at
            FROM incidents
            WHERE record_id IN ({placeholders})
            ORDER BY created_at DESC
        """, db, params=tuple(record_ids))

    # Explain whether the ID matches multiple records
    if len(claims) > 1:
        summary = (
            f"Found {len(claims)} records sharing "
            f"Claim ID {claim_id}. "
            "Review their unique record IDs."
        )
    else:
        summary = (
            f"Found claim {claim_id}. "
            f"Current status: {claims.iloc[0]['status']}."
        )

    summary += f"\nLinked incident tickets: {len(tickets)}"

    return claims, tickets, summary


# Test with an existing synthetic claim
claims_result, tickets_result, message = search_claim(
    "CLM100027"
)

print(message)
display(claims_result)
display(tickets_result)


Found claim CLM100027. Current status: FAILED.
Linked incident tickets: 1


,record_id,claim_id,patient_id,provider,insurance_company,procedure_code,diagnosis_code,claim_amount,status,error_reason
0,27,CLM100027,PAT4295,Lone Star Healthcare,Blue Cross Blue Shield,93000,J06.9,-100.0,FAILED,Invalid Claim Amount


,ticket_id,record_id,priority,ticket_status,error_reason,created_at,resolved_at
0,INC100027,27,HIGH,OPEN,Invalid Claim Amount,2026-10-19T17:30:57.619166,None


In [45]:

# STEP 59C: Incident Evidence Timeline

def get_incident_timeline(ticket_id):

    ticket_id = str(ticket_id).strip().upper()

    if not ticket_id:
        return pd.DataFrame(), "Enter a ticket ID."

    with sqlite3.connect(DATABASE_PATH) as db:

        ticket = db.execute("""
            SELECT ticket_id
            FROM incidents
            WHERE ticket_id = ?
        """, (ticket_id,)).fetchone()

        if not ticket:
            return pd.DataFrame(), "Ticket not found."

        events = []

        incident = pd.read_sql_query("""
            SELECT created_at, resolved_at,
                   ticket_status, error_reason
            FROM incidents
            WHERE ticket_id = ?
        """, db, params=(ticket_id,)).iloc[0]

        events.append({
            "timestamp": incident["created_at"],
            "event": "INCIDENT_CREATED",
            "details": incident["error_reason"]
        })

        replay_events = pd.read_sql_query("""
            SELECT attempted_at, replay_status, notes
            FROM replay_history
            WHERE ticket_id = ?
        """, db, params=(ticket_id,))

        for _, row in replay_events.iterrows():
            events.append({
                "timestamp": row["attempted_at"],
                "event": "REPLAY_" + row["replay_status"],
                "details": row["notes"]
            })

        audit_events = pd.read_sql_query("""
            SELECT action_time, action,
                   performed_by, notes
            FROM incident_audit_log
            WHERE ticket_id = ?
        """, db, params=(ticket_id,))

        for _, row in audit_events.iterrows():
            events.append({
                "timestamp": row["action_time"],
                "event": row["action"],
                "details": (
                    f"{row['performed_by']}: {row['notes']}"
                )
            })

    timeline = pd.DataFrame(events)

    timeline["timestamp"] = pd.to_datetime(
        timeline["timestamp"], utc=True
    )

    timeline = timeline.sort_values("timestamp")

    return timeline, f"Found {len(timeline)} recorded events."


# Test a ticket from the earlier replay demonstration
timeline, timeline_message = get_incident_timeline(
    "INC100026"
)

print(timeline_message)
display(timeline)


Found 3 recorded events.


,timestamp,event,details
0,2026-10-19 17:30:57.619089+00:00,INCIDENT_CREATED,Invalid Claim Amount
1,2026-10-19 17:30:58.327508+00:00,REPLAY_SUCCESS,Validated correction to claim_amount
2,2026-10-19 17:30:58.327508+00:00,CLAIM_CORRECTED_AND_VALIDATED,Demo Support Engineer: Corrected claim_amount;...


In [46]:

# STEP 60A: Intelligent Root-Cause Investigation Engine

import sqlite3
import pandas as pd
from pathlib import Path

DATABASE_PATH = str(Path("claimguard.db").resolve())

def investigate_root_cause(ticket_id):

    ticket_id = str(ticket_id).strip().upper()

    if not ticket_id:
        return "Please enter an incident ticket ID.", pd.DataFrame()

    with sqlite3.connect(DATABASE_PATH) as db:

        incident = pd.read_sql_query("""
            SELECT
                i.ticket_id,
                i.record_id,
                i.claim_id,
                i.error_reason,
                i.priority,
                i.ticket_status,
                i.created_at,
                i.resolved_at,
                c.patient_id,
                c.provider,
                c.insurance_company,
                c.procedure_code,
                c.claim_amount,
                c.status AS claim_status
            FROM incidents i
            JOIN claims c
                ON i.record_id = c.record_id
            WHERE i.ticket_id = ?
        """, db, params=(ticket_id,))

        if incident.empty:
            return "Incident not found.", pd.DataFrame()

        row = incident.iloc[0]

        original_error = str(row["error_reason"])

        # Reuse our existing recovery policy engine
        analysis = recommend_recovery(original_error)

        # Build an evidence-based explanation
        evidence = []

        if "Missing Patient ID" in original_error:
            evidence.append(
                "The original validation detected a missing "
                "patient identifier."
            )

        if "Invalid Claim Amount" in original_error:
            evidence.append(
                "The original validation detected a "
                "non-positive claim amount."
            )

        if "Invalid Procedure Code" in original_error:
            evidence.append(
                "The procedure code failed the configured "
                "validation rules."
            )

        if "Duplicate Claim ID" in original_error:
            evidence.append(
                "Multiple transaction records shared "
                "the same external claim ID."
            )

        if not evidence:
            evidence.append(
                "The incident requires additional investigation."
            )

        report = f"""
# ClaimGuard AI — Root-Cause Investigation

**Ticket:** {row['ticket_id']}

**Record ID:** {row['record_id']}

**Claim ID:** {row['claim_id']}

**Priority:** {row['priority']}

**Incident Status:** {row['ticket_status']}

**Current Claim Status:** {row['claim_status']}

### Original Detected Issue
{original_error}

### Validation Evidence
{" ".join(evidence)}

### Root-Cause Category
{analysis['category']}

### Recommended Recovery Strategy
{analysis['strategy']}

### Recommended Action
{analysis['recommendation']}

### Investigation Note
These findings are based on recorded validation errors.
The underlying business cause has not been independently verified.
"""

        related_claims = pd.read_sql_query("""
            SELECT
                record_id,
                claim_id,
                provider,
                claim_amount,
                status,
                error_reason
            FROM claims
            WHERE claim_id = ?
            ORDER BY record_id
        """, db, params=(row["claim_id"],))

    return report, related_claims


print("Root-cause investigation engine initialized!")

report, related = investigate_root_cause("INC100027")
print(report)
display(related)


Root-cause investigation engine initialized!

# ClaimGuard AI — Root-Cause Investigation

**Ticket:** INC100027

**Record ID:** 27

**Claim ID:** CLM100027

**Priority:** HIGH

**Incident Status:** OPEN

**Current Claim Status:** FAILED

### Original Detected Issue
Invalid Claim Amount

### Validation Evidence
The original validation detected a non-positive claim amount.

### Root-Cause Category
INVALID_AMOUNT

### Recommended Recovery Strategy
MANUAL_CORRECTION

### Recommended Action
Verify the billed amount against the source record before correcting it.

### Investigation Note
These findings are based on recorded validation errors.
The underlying business cause has not been independently verified.



,record_id,claim_id,provider,claim_amount,status,error_reason
0,27,CLM100027,Lone Star Healthcare,-100.0,FAILED,Invalid Claim Amount


In [47]:

# STEP 60B: Historical Incident Pattern Analysis

def find_similar_incidents(ticket_id):

    ticket_id = str(ticket_id).strip().upper()

    with sqlite3.connect(DATABASE_PATH) as db:

        target = db.execute("""
            SELECT error_reason
            FROM incidents
            WHERE ticket_id = ?
        """, (ticket_id,)).fetchone()

        if target is None:
            return pd.DataFrame(), "Incident not found."

        error_reason = target[0]

        similar = pd.read_sql_query("""
            SELECT
                ticket_id,
                claim_id,
                priority,
                ticket_status,
                created_at
            FROM incidents
            WHERE error_reason = ?
              AND ticket_id != ?
            ORDER BY created_at DESC
            LIMIT 20
        """, db, params=(error_reason, ticket_id))

        total_similar = db.execute("""
            SELECT COUNT(*)
            FROM incidents
            WHERE error_reason = ?
              AND ticket_id != ?
        """, (error_reason, ticket_id)).fetchone()[0]

    summary = (
        f"Found {total_similar} other incidents "
        f"with the same recorded error: {error_reason}"
    )

    return similar, summary


similar, summary = find_similar_incidents("INC100027")

print(summary)
display(similar)


Found 24 other incidents with the same recorded error: Invalid Claim Amount


,ticket_id,claim_id,priority,ticket_status,created_at
0,INC100050,CLM100050,HIGH,OPEN,2026-10-19T17:30:57.623074
1,INC100049,CLM100049,HIGH,OPEN,2026-10-19T17:30:57.622969
2,INC100048,CLM100048,HIGH,OPEN,2026-10-19T17:30:57.622878
3,INC100047,CLM100047,HIGH,OPEN,2026-10-19T17:30:57.622804
4,INC100046,CLM100046,HIGH,OPEN,2026-10-19T17:30:57.622733
5,INC100045,CLM100045,HIGH,OPEN,2026-10-19T17:30:57.622654
6,INC100044,CLM100044,HIGH,OPEN,2026-10-19T17:30:57.622583
7,INC100043,CLM100043,HIGH,OPEN,2026-10-19T17:30:57.622511
8,INC100042,CLM100042,HIGH,OPEN,2026-10-19T17:30:57.622435
9,INC100041,CLM100041,HIGH,OPEN,2026-10-19T17:30:57.622358


In [48]:

# STEP 61A: Live SLA Monitoring

import sqlite3
import pandas as pd
from pathlib import Path

DATABASE_PATH = str(Path("claimguard.db").resolve())

SLA_TARGETS = {
    "HIGH": 4,
    "MEDIUM": 12,
    "LOW": 24
}

def get_live_sla_data():

    with sqlite3.connect(DATABASE_PATH) as db:
        incidents = pd.read_sql_query("""
            SELECT ticket_id, record_id, claim_id,
                   priority, ticket_status,
                   error_reason, created_at, resolved_at
            FROM incidents
        """, db)

    incidents["created_at"] = pd.to_datetime(
        incidents["created_at"], utc=True
    )

    incidents["resolved_at"] = pd.to_datetime(
        incidents["resolved_at"], utc=True
    )

    now = pd.Timestamp.now(tz="UTC")

    incidents["sla_hours"] = (
        incidents["priority"].map(SLA_TARGETS)
    )

    incidents["deadline"] = (
        incidents["created_at"] +
        pd.to_timedelta(incidents["sla_hours"], unit="h")
    )

    def classify(row):

        if pd.isna(row["deadline"]):
            return "UNKNOWN"

        if row["ticket_status"] == "RESOLVED":
            if pd.isna(row["resolved_at"]):
                return "UNKNOWN"

            return (
                "RESOLVED_ON_TIME"
                if row["resolved_at"] <= row["deadline"]
                else "RESOLVED_LATE"
            )

        if row["ticket_status"] != "OPEN":
            return "NOT_APPLICABLE"

        remaining = (
            row["deadline"] - now
        ).total_seconds() / 3600

        if remaining < 0:
            return "BREACHED"
        elif remaining <= 1:
            return "AT_RISK"
        else:
            return "WITHIN_SLA"

    incidents["sla_status"] = incidents.apply(
        classify, axis=1
    )

    return incidents


sla_live = get_live_sla_data()

display(
    sla_live[
        ["ticket_id", "priority", "ticket_status",
         "deadline", "sla_status"]
    ].head(10)
)

print("Live SLA monitoring initialized!")


,ticket_id,priority,ticket_status,deadline,sla_status
0,INC100001,LOW,RESOLVED,2026-10-20 17:30:57.614283+00:00,RESOLVED_ON_TIME
1,INC100002,LOW,OPEN,2026-10-20 17:30:57.614444+00:00,WITHIN_SLA
2,INC100003,LOW,OPEN,2026-10-20 17:30:57.614573+00:00,WITHIN_SLA
3,INC100004,LOW,OPEN,2026-10-20 17:30:57.614714+00:00,WITHIN_SLA
4,INC100005,LOW,OPEN,2026-10-20 17:30:57.614825+00:00,WITHIN_SLA
5,INC100006,LOW,OPEN,2026-10-20 17:30:57.614933+00:00,WITHIN_SLA
6,INC100007,LOW,OPEN,2026-10-20 17:30:57.617108+00:00,WITHIN_SLA
7,INC100008,LOW,OPEN,2026-10-20 17:30:57.617293+00:00,WITHIN_SLA
8,INC100009,LOW,OPEN,2026-10-20 17:30:57.617410+00:00,WITHIN_SLA
9,INC100010,LOW,OPEN,2026-10-20 17:30:57.617520+00:00,WITHIN_SLA


Live SLA monitoring initialized!


In [49]:

# STEP 62A: Recovery Analytics Backend

import sqlite3
import pandas as pd
from pathlib import Path

DATABASE_PATH = str(Path("claimguard.db").resolve())

def get_recovery_analytics():

    with sqlite3.connect(DATABASE_PATH) as db:

        jobs = pd.read_sql_query("""
            SELECT
                j.record_id,
                c.claim_id,
                j.scenario,
                j.job_status,
                j.attempts,
                j.last_error,
                j.updated_at
            FROM processing_jobs j
            JOIN claims c
                ON j.record_id = c.record_id
            ORDER BY j.record_id
        """, db)

        attempts = pd.read_sql_query("""
            SELECT
                a.record_id,
                c.claim_id,
                a.attempt_number,
                a.outcome,
                a.details,
                a.attempted_at
            FROM processing_attempt_log a
            JOIN claims c
                ON a.record_id = c.record_id
            ORDER BY a.record_id, a.attempt_number
        """, db)

        duplicate_receipts = db.execute("""
            SELECT COUNT(*)
            FROM (
                SELECT record_id
                FROM processing_receipts
                GROUP BY record_id
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

    total = len(jobs)

    completed = int(
        (jobs["job_status"] == "COMPLETED").sum()
    )

    recovered = int(
        (jobs["job_status"] == "RECOVERED").sum()
    )

    escalated = int(
        (jobs["job_status"] == "ESCALATED").sum()
    )

    return (
        jobs,
        attempts,
        total,
        completed,
        recovered,
        escalated,
        duplicate_receipts
    )


result = get_recovery_analytics()

print("Recovery Analytics Ready!")
print("Total Jobs:", result[2])
print("Completed:", result[3])
print("Recovered:", result[4])
print("Escalated:", result[5])
print("Duplicate Receipts:", result[6])


Recovery Analytics Ready!
Total Jobs: 12
Completed: 6
Recovered: 3
Escalated: 3
Duplicate Receipts: 0


In [50]:

# STEP 62B: Investigate Processing Attempts

def investigate_processing_job(record_id):

    try:
        record_id = int(record_id)
    except (TypeError, ValueError):
        return (
            "Please enter a valid numeric Record ID.",
            pd.DataFrame()
        )

    with sqlite3.connect(DATABASE_PATH) as db:

        job = pd.read_sql_query("""
            SELECT
                j.record_id,
                c.claim_id,
                j.scenario,
                j.job_status,
                j.attempts,
                j.last_error
            FROM processing_jobs j
            JOIN claims c
                ON j.record_id = c.record_id
            WHERE j.record_id = ?
        """, db, params=(record_id,))

        history = pd.read_sql_query("""
            SELECT
                attempt_number,
                outcome,
                details,
                attempted_at
            FROM processing_attempt_log
            WHERE record_id = ?
            ORDER BY attempt_number
        """, db, params=(record_id,))

        receipt = db.execute("""
            SELECT processed_at
            FROM processing_receipts
            WHERE record_id = ?
        """, (record_id,)).fetchone()

    if job.empty:
        return "Processing job not found.", pd.DataFrame()

    row = job.iloc[0]

    receipt_status = (
        f"Recorded at {receipt[0]}"
        if receipt else "No successful processing receipt"
    )

    summary = f"""
### Transaction Investigation

**Record ID:** {row['record_id']}

**Claim ID:** {row['claim_id']}

**Simulated Failure Scenario:** {row['scenario']}

**Processing Status:** {row['job_status']}

**Attempts:** {row['attempts']}

**Last Error:** {row['last_error']}

**Processing Receipt:** {receipt_status}
"""

    return summary, history


# Test an existing processing job
first_record = int(result[0].iloc[0]["record_id"])

summary, history = investigate_processing_job(first_record)

print(summary)
display(history)



### Transaction Investigation

**Record ID:** 26

**Claim ID:** CLM100026

**Simulated Failure Scenario:** TIMEOUT_THEN_SUCCESS

**Processing Status:** COMPLETED

**Attempts:** 2

**Last Error:** None

**Processing Receipt:** Recorded at 2026-10-19T17:30:59.453248+00:00



,attempt_number,outcome,details,attempted_at
0,1,TIMEOUT,Scenario: TIMEOUT_THEN_SUCCESS,2026-10-19T17:30:59.391603+00:00
1,2,SUCCESS,Scenario: TIMEOUT_THEN_SUCCESS,2026-10-19T17:30:59.461271+00:00


In [51]:

# STEP 63A: Upgrade Incident Management Schema

import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

DATABASE_PATH = str(Path("claimguard.db").resolve())

def current_utc():
    return datetime.now(timezone.utc).isoformat()

with sqlite3.connect(DATABASE_PATH) as db:

    columns = {
        row[1]
        for row in db.execute(
            "PRAGMA table_info(incidents)"
        ).fetchall()
    }

    new_columns = {
        "resolution_notes": "TEXT",
        "resolved_by": "TEXT",
        "escalated_at": "TEXT",
        "escalated_by": "TEXT"
    }

    for column, data_type in new_columns.items():
        if column not in columns:
            db.execute(
                f"ALTER TABLE incidents "
                f"ADD COLUMN {column} {data_type}"
            )
            print("Added:", column)

    db.execute("""
        CREATE TABLE IF NOT EXISTS incident_audit_log (
            audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticket_id TEXT NOT NULL,
            action TEXT NOT NULL,
            performed_by TEXT NOT NULL,
            action_time TEXT NOT NULL,
            notes TEXT
        )
    """)

print("Incident management database upgraded!")


Added: resolution_notes
Added: resolved_by
Added: escalated_at
Added: escalated_by
Incident management database upgraded!


In [52]:

# STEP 63B: Controlled Incident Action Engine

def update_incident(ticket_id, engineer, action, notes):

    ticket_id = str(ticket_id).strip().upper()
    engineer = str(engineer).strip()
    notes = str(notes).strip()
    action = str(action).strip().upper()

    if not ticket_id or not engineer or not notes:
        return "Ticket ID, engineer name, and notes are required."

    if action not in ("ESCALATE", "RESOLVE"):
        return "Invalid action. Choose ESCALATE or RESOLVE."

    if len(notes) < 15:
        return "Please provide at least 15 characters of documentation."

    with sqlite3.connect(DATABASE_PATH) as db:

        db.execute("BEGIN IMMEDIATE")

        ticket = db.execute("""
            SELECT
                i.ticket_status,
                i.record_id,
                c.status
            FROM incidents i
            JOIN claims c
                ON i.record_id = c.record_id
            WHERE i.ticket_id = ?
        """, (ticket_id,)).fetchone()

        if ticket is None:
            return "Incident not found."

        ticket_status, record_id, claim_status = ticket

        if ticket_status == "RESOLVED":
            return "This incident is already resolved."

        timestamp = current_utc()

        if action == "RESOLVE":

            if claim_status != "VALIDATED":
                return (
                    "Resolution blocked: claim has not passed "
                    "validation. Investigate or correct it first."
                )

            if ticket_status not in ("OPEN", "ESCALATED"):
                return "This incident cannot be resolved."

            db.execute("""
                UPDATE incidents
                SET ticket_status = 'RESOLVED',
                    resolved_at = ?,
                    resolved_by = ?,
                    resolution_notes = ?
                WHERE ticket_id = ?
            """, (
                timestamp, engineer, notes, ticket_id
            ))

            audit_action = "INCIDENT_RESOLVED"

        else:

            if ticket_status != "OPEN":
                return "Only OPEN incidents can be escalated."

            db.execute("""
                UPDATE incidents
                SET ticket_status = 'ESCALATED',
                    escalated_at = ?,
                    escalated_by = ?
                WHERE ticket_id = ?
            """, (
                timestamp, engineer, ticket_id
            ))

            audit_action = "INCIDENT_ESCALATED"

        db.execute("""
            INSERT INTO incident_audit_log
            (ticket_id, action, performed_by,
             action_time, notes)
            VALUES (?, ?, ?, ?, ?)
        """, (
            ticket_id,
            audit_action,
            engineer,
            timestamp,
            notes
        ))

    return (
        f"SUCCESS: {ticket_id} updated. "
        f"Action: {action}. Audit event recorded."
    )


print("Controlled incident action engine initialized!")


Controlled incident action engine initialized!


In [53]:

# STEP 63C: Preview Incident Before Action

def preview_incident(ticket_id):

    ticket_id = str(ticket_id).strip().upper()

    with sqlite3.connect(DATABASE_PATH) as db:

        incident = pd.read_sql_query("""
            SELECT
                i.ticket_id,
                i.record_id,
                i.priority,
                i.ticket_status,
                i.error_reason,
                i.created_at,
                i.resolved_at,
                c.claim_id,
                c.provider,
                c.status AS claim_status
            FROM incidents i
            JOIN claims c
                ON i.record_id = c.record_id
            WHERE i.ticket_id = ?
        """, db, params=(ticket_id,))

    if incident.empty:
        return pd.DataFrame(), "Incident not found."

    row = incident.iloc[0]

    if row["ticket_status"] == "RESOLVED":
        message = "Incident already resolved."

    elif row["claim_status"] == "VALIDATED":
        message = (
            "Claim is validated. Incident may be eligible "
            "for resolution after reviewing the evidence."
        )

    else:
        message = (
            "Claim remains unvalidated. "
            "Resolution is blocked; investigation or "
            "correction is required."
        )

    return incident, message


# Preview an existing open incident
preview, message = preview_incident("INC100027")

print(message)
display(preview)


Claim remains unvalidated. Resolution is blocked; investigation or correction is required.


,ticket_id,record_id,priority,ticket_status,error_reason,created_at,resolved_at,claim_id,provider,claim_status
0,INC100027,27,HIGH,OPEN,Invalid Claim Amount,2026-10-19T17:30:57.619166,None,CLM100027,Lone Star Healthcare,FAILED


In [54]:

# FIX: Restore SLA Dashboard Function

import sqlite3
import pandas as pd
import plotly.express as px

def refresh_sla_dashboard():

    with sqlite3.connect(DATABASE_PATH) as db:
        data = pd.read_sql_query("""
            SELECT ticket_id, claim_id, priority,
                   ticket_status, created_at, resolved_at
            FROM incidents
        """, db)

    now = pd.Timestamp.now(tz="UTC")

    data["created_at"] = pd.to_datetime(
        data["created_at"], utc=True
    )

    data["resolved_at"] = pd.to_datetime(
        data["resolved_at"], utc=True
    )

    targets = {"HIGH": 4, "MEDIUM": 12, "LOW": 24}

    data["deadline"] = (
        data["created_at"] +
        pd.to_timedelta(data["priority"].map(targets), unit="h")
    )

    def status(row):
        if row["ticket_status"] == "RESOLVED":
            if pd.isna(row["resolved_at"]):
                return "UNKNOWN"
            return ("RESOLVED_ON_TIME"
                    if row["resolved_at"] <= row["deadline"]
                    else "RESOLVED_LATE")

        if row["ticket_status"] != "OPEN":
            return "NOT_APPLICABLE"

        hours = (row["deadline"] - now).total_seconds() / 3600

        if hours < 0:
            return "BREACHED"
        if hours <= 1:
            return "AT_RISK"
        return "WITHIN_SLA"

    data["sla_status"] = data.apply(status, axis=1)

    urgent = data[
        data["sla_status"].isin(["BREACHED", "AT_RISK"])
    ].copy()

    urgent["deadline"] = urgent["deadline"].astype(str)

    summary = data["sla_status"].value_counts().reset_index()
    summary.columns = ["sla_status", "count"]

    fig = px.bar(
        summary,
        x="sla_status",
        y="count",
        color="sla_status",
        text="count",
        title="Incident SLA Status"
    )

    return (
        str((data["sla_status"] == "BREACHED").sum()),
        str((data["sla_status"] == "AT_RISK").sum()),
        fig,
        urgent[[
            "ticket_id", "claim_id", "priority",
            "sla_status", "deadline"
        ]]
    )

print("SLA Dashboard function restored!")


SLA Dashboard function restored!


In [55]:
# Final dashboard callbacks: separate connections per event

def dashboard_overview():
    return get_professional_dashboard()[:6]

def dashboard_recovery():
    jobs, attempts, total, completed, recovered, escalated, duplicates = get_recovery_analytics()
    if jobs.empty:
        fig = px.bar(title="No processing jobs")
    else:
        grouped = jobs.groupby("job_status").size().reset_index(name="count")
        fig = px.bar(grouped, x="job_status", y="count", color="job_status",
                     text="count", title="Automated Recovery Results")
    fig.update_layout(template="plotly_white", height=380)
    return fig, jobs

def dashboard_sla():
    # refresh_sla_dashboard opens a fresh DB connection
    return refresh_sla_dashboard()

def dashboard_resolution(ticket_id, engineer, action, notes):
    return update_incident(ticket_id, engineer, action, notes)

# Lightweight backend checks (no database changes)
assert len(dashboard_overview()) == 6
assert len(dashboard_recovery()) == 2
assert len(dashboard_sla()) == 4
assert len(search_claim("CLM100027")) == 3
assert len(investigate_root_cause("INC100027")) == 2
print("PASS: All five read-only dashboard callbacks")


PASS: All five read-only dashboard callbacks


In [56]:
# STEP 64: ONE unified application (replaces all debug dashboards)
with gr.Blocks(title="ClaimGuard AI | Support Operations Center") as final_app:
    gr.Markdown("# 🛡️ ClaimGuard AI — Support Operations Center")
    gr.Markdown("**Synthetic healthcare data · Educational portfolio demo**")

    with gr.Tab("Operations"):
        with gr.Row():
            m_total = gr.Textbox(label="Total Claims", interactive=False)
            m_valid = gr.Textbox(label="Validated", interactive=False)
            m_failed = gr.Textbox(label="Failed", interactive=False)
            m_open = gr.Textbox(label="Open Incidents", interactive=False)
        with gr.Row():
            chart_claims = gr.Plot(label="Claim Validation")
            chart_priority = gr.Plot(label="Incident Priority")
        gr.Button("Refresh Dashboard", variant="primary").click(
            dashboard_overview, inputs=[],
            outputs=[m_total,m_valid,m_failed,m_open,chart_claims,chart_priority], queue=False)

    with gr.Tab("Claim Search"):
        claim_search_id = gr.Textbox(label="Claim ID", value="CLM100027")
        search_status = gr.Textbox(label="Search Summary", interactive=False)
        search_claims = gr.Dataframe(label="Matching Records", interactive=False)
        search_tickets = gr.Dataframe(label="Linked Incidents", interactive=False)
        gr.Button("Search", variant="primary").click(search_claim, [claim_search_id],
            [search_claims, search_tickets, search_status], queue=False)

    with gr.Tab("Root Cause"):
        root_id = gr.Textbox(label="Incident ID", value="INC100027")
        root_report = gr.Markdown()
        root_related = gr.Dataframe(label="Related Claims", interactive=False)
        gr.Button("Investigate", variant="primary").click(investigate_root_cause,
            [root_id], [root_report, root_related], queue=False)

    with gr.Tab("SLA Monitoring"):
        with gr.Row():
            breached = gr.Textbox(label="Breached", interactive=False)
            at_risk = gr.Textbox(label="At Risk", interactive=False)
        sla_chart = gr.Plot()
        sla_urgent = gr.Dataframe(label="Urgent Incidents", interactive=False)
        gr.Button("Check SLA", variant="primary").click(dashboard_sla, [],
            [breached, at_risk, sla_chart, sla_urgent], queue=False)

    with gr.Tab("Recovery Monitoring"):
        recovery_plot = gr.Plot()
        recovery_jobs = gr.Dataframe(label="Processing Jobs", interactive=False)
        gr.Button("View Recovery", variant="primary").click(dashboard_recovery, [],
            [recovery_plot, recovery_jobs], queue=False)

    with gr.Tab("Incident Actions"):
        action_ticket = gr.Textbox(label="Ticket ID", value="INC100027")
        action_engineer = gr.Textbox(label="Demo Engineer Name (not authentication)")
        action_kind = gr.Radio(["ESCALATE", "RESOLVE"], label="Action", value="ESCALATE")
        action_notes = gr.Textbox(label="Investigation Notes (15+ characters)", lines=3)
        action_feedback = gr.Textbox(label="Action Result", interactive=False)
        gr.Markdown("⚠️ Submitting an action changes the SQLite incident record.")
        gr.Button("Submit Action", variant="primary").click(dashboard_resolution,
            [action_ticket, action_engineer, action_kind, action_notes],
            [action_feedback], queue=False)

final_app.launch(inline=True, share=False, show_error=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [57]:
# Final non-destructive checks and backup
from pathlib import Path
with sqlite3.connect(DATABASE_PATH) as db:
    summary = pd.read_sql_query("SELECT status, COUNT(*) AS n FROM claims GROUP BY status", db)
    tickets = pd.read_sql_query("SELECT ticket_status, COUNT(*) AS n FROM incidents GROUP BY ticket_status", db)
    receipts = db.execute("SELECT COUNT(*) FROM processing_receipts").fetchone()[0]
    duplicates = db.execute("SELECT COUNT(*) FROM (SELECT record_id FROM processing_receipts GROUP BY record_id HAVING COUNT(*) > 1)").fetchone()[0]
print("Claims:")
display(summary)
print("Incident statuses:")
display(tickets)
print("Processing receipts:", receipts, "Duplicate receipt IDs:", duplicates)
assert duplicates == 0
assert summary.n.sum() == 1000
print("PASS: final data integrity checks")
# Download database separately if you want to preserve any changes made through Gradio.


Claims:


,status,n
0,FAILED,99
1,VALIDATED,901


Incident statuses:


,ticket_status,n
0,OPEN,98
1,RESOLVED,2


Processing receipts: 9 Duplicate receipt IDs: 0
PASS: final data integrity checks
